# Ontology Enrichment — Structured Output Testing

This notebook tests the LLM-powered ontology enrichment system:
- `OntologySchema` — structured output schema
- `OntologyEnrichmentStep` — enrichment pipeline step
- System prompt performance

**Requirements**: Qdrant running, `ANTHROPIC_API_KEY` set in `.env`

## 1. Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from dotenv import load_dotenv
load_dotenv()

from src.ingestion.ontology_enricher import (
    OntologySchema,
    OntologyEnrichmentStep,
    ONTOLOGY_SYSTEM_PROMPT,
)
from src.core.llm import get_llm
from src.ingestion.types import Chunk, ChunkMetadata, DataSourceCategory

print("✓ Imports OK")
print(f"System prompt length: {len(ONTOLOGY_SYSTEM_PROMPT)} chars")

## 2. Inspect the Schema & System Prompt

In [ ]:
# Print the full system prompt
print("=" * 60)
print("ONTOLOGY SYSTEM PROMPT")
print("=" * 60)
print(ONTOLOGY_SYSTEM_PROMPT)

print("\n" + "=" * 60)
print("ONTOLOGY SCHEMA FIELDS")
print("=" * 60)
for name, field in OntologySchema.model_fields.items():
    print(f"\n{name}:")
    print(f"  description: {field.description}")

## 3. Test Structured Output with a Single Chunk

In [ ]:
# Sample SEC filing excerpt (Risk Factors — AI related)
sample_chunk_content = """
Item 1A. Risk Factors

Our future growth is dependent upon our ability to continue to develop and innovate with respect to AI technologies. We have invested heavily in developing AI-powered features across our product portfolio, including generative AI capabilities in our enterprise software platforms. These investments may not result in sufficient revenue growth to offset the costs. Competitive pressures from well-capitalized technology companies with significant AI capabilities could materially adversely affect our business.

Furthermore, the regulatory landscape for AI remains uncertain. We may face scrutiny from data privacy regulators regarding our collection and processing of training data for our AI models. Potential legislation governing AI decision-making systems could require us to modify our product offerings or face compliance costs.

We depend on a limited number of third-party GPU providers for our AI compute requirements. Any disruption in the supply chain for AI hardware could materially adversely affect our ability to serve our customers.
"""

# Create a mock chunk
chunk_metadata = ChunkMetadata(
    chunk_id="test_chunk_001",
    document_id="NVDA_10K_2024",
    source_category=DataSourceCategory.PUBLIC_SEC,
    source_detail="nvda_10k_2024",
    content_type="risk_factors",
    sec_section="Item 1A",
)
chunk = Chunk(content=sample_chunk_content, metadata=chunk_metadata)
print(f"Chunk content length: {len(chunk.content)} chars")

In [ ]:
# Run enrichment
step = OntologyEnrichmentStep()
result_chunk = step.process(chunk)

print("=" * 60)
print("ONTOLOGY ENRICHMENT RESULTS")
print("=" * 60)
print(f"\nconcepts ({len(result_chunk.metadata.concepts)}):")
for c in result_chunk.metadata.concepts:
    print(f"  • {c}")

print(f"\nstrategic_themes ({len(result_chunk.metadata.strategic_themes)}):")
for t in result_chunk.metadata.strategic_themes:
    print(f"  • {t}")

print(f"\ndepartments ({len(result_chunk.metadata.departments)}):")
for d in result_chunk.metadata.departments:
    print(f"  • {d}")

print(f"\nontology_enrichment_failed: {result_chunk.metadata.ontology_enrichment_failed}")

## 4. Test with a Financial Statements Excerpt

In [ ]:
# Sample — Financial Statements
financials_chunk_content = """
Item 8. Financial Statements and Supplementary Data

Revenue for fiscal year 2024 was $60.9 billion, an increase of 122% compared to $27.4 billion in fiscal year 2023. Data center revenue was $47.5 billion, up 409% year-over-year, driven by surging demand for H100 and H200 GPU clusters from hyperscale customers. Gaming revenue was $2.9 billion, down 4% year-over-year, reflecting a decline in consumer GPU demand. Professional Visualization revenue was $1.6 billion, up 1% year-over-year.

Gross margin was 74.6% for fiscal 2024, compared to 56.9% for fiscal 2023. The increase was primarily driven by a shift toward higher-margin data center products and improved manufacturing efficiency. Operating income was $32.8 billion, up 182% year-over-year. Net income was $29.8 billion, or $1.21 per diluted share, compared to $4.4 billion, or $0.18 per diluted share, in fiscal 2023.

As of January 28, 2024, we had cash and marketable securities of $29.1 billion and total debt of $9.7 billion. Cash flow from operations was $26.5 billion for fiscal 2024, compared to $7.3 billion for fiscal 2023.
"""

chunk2_metadata = ChunkMetadata(
    chunk_id="test_chunk_002",
    document_id="NVDA_10K_2024",
    source_category=DataSourceCategory.PUBLIC_SEC,
    source_detail="nvda_10k_2024",
    content_type="financial_statements",
    sec_section="Item 8",
    fiscal_year=2024,
)
chunk2 = Chunk(content=financials_chunk_content, metadata=chunk2_metadata)

result_chunk2 = step.process(chunk2)

print(f"concepts: {result_chunk2.metadata.concepts}")
print(f"strategic_themes: {result_chunk2.metadata.strategic_themes}")
print(f"departments: {result_chunk2.metadata.departments}")
print(f"failed: {result_chunk2.metadata.ontology_enrichment_failed}")

## 5. Test Batch Processing

In [ ]:
chunks = [chunk, chunk2]
results = step.process_batch(chunks)

print(f"Processed {len(results)} chunks")
for i, c in enumerate(results):
    print(f"\nChunk {i+1} ({c.metadata.chunk_id}):")
    print(f"  concepts: {c.metadata.concepts}")
    print(f"  themes:   {c.metadata.strategic_themes}")
    print(f"  depts:    {c.metadata.departments}")

## 6. Test with Ingestion Pipeline (Full E2E)

In [ ]:
# Requires Qdrant running + existing NVDA filing ingested
from src.ingestion import create_ingestion_pipeline
from src.core import VectorStore

vs = VectorStore()
print(f"Vector store count: {vs.count}")

# Ingest a new filing WITH ontology enabled
pipeline = create_ingestion_pipeline(enable_ontology=True)
print(f"Ontology step: {pipeline._ontology_step}")

# Uncomment to run:
# count = pipeline.ingest_sec_filing("NVDA", 2024, "10-K")
# print(f"Ingested {count} chunks with ontology enrichment")

## 7. Verify Qdrant Payload (after enrichment)

In [ ]:
# After running ingest, verify the metadata is stored
vs = VectorStore()

# Find chunks for NVDA filing
docs = vs.get_by_original_id("NVDA_10K_2024")
print(f"Found {len(docs)} chunks for NVDA_10K_2024")

if docs:
    # Show first chunk
    first = docs[0]
    meta = first.metadata
    print(f"\nFirst chunk metadata:")
    print(f"  concepts:                 {meta.get('concepts', '')}")
    print(f"  strategic_themes:         {meta.get('strategic_themes', '')}")
    print(f"  departments:              {meta.get('departments', '')}")
    print(f"  ontology_enrichment_failed: {meta.get('ontology_enrichment_failed', '')}")

    # Show last chunk
    last = docs[-1]
    last_meta = last.metadata
    print(f"\nLast chunk metadata:")
    print(f"  concepts:                 {last_meta.get('concepts', '')}")
    print(f"  strategic_themes:         {last_meta.get('strategic_themes', '')}")
    print(f"  departments:              {last_meta.get('departments', '')}")

## 8. Test Script Verification

In [ ]:
# Run the CLI test script
import subprocess
result = subprocess.run(
    ["uv", "run", "python", "scripts/test_ontology.py", "--stats"],
    capture_output=True,
    text=True,
    cwd=Path.cwd().parent,
)
print("STDOUT:", result.stdout)
print("STDERR:", result.stderr[:500] if result.stderr else "")

## 9. Retry / Failure Behavior Test

In [ ]:
# Test that failure sets the flag correctly
# Create a step where we mock the LLM to fail once then succeed
import unittest.mock as mock

step2 = OntologyEnrichmentStep()

# Patch _get_client to return a client that fails then succeeds
fail_count = [0]

def failing_invoke(text):
    fail_count[0] += 1
    if fail_count[0] == 1:
        raise Exception("Simulated transient failure")
    return OntologySchema(
        concepts=["test_concept"],
        strategic_themes=["test_theme"],
        departments=["test_dept"],
    )

with mock.patch.object(step2, '_get_client') as mock_get:
    mock_client = mock.MagicMock()
    mock_client.invoke = failing_invoke
    mock_get.return_value = mock_client

    result = step2.process(chunk)

print(f"Attempt count: {fail_count[0]}")
print(f"concepts: {result.metadata.concepts}")
print(f"ontology_enrichment_failed: {result.metadata.ontology_enrichment_failed}")
print(f"Retry worked: {'✓' if result.metadata.concepts == ['test_concept'] else '✗'}")